# Phase 2 — `player_match_stats` (the atomic table)

This is the **most important table in the project**. Every rating, every form
data point, and every validation check is built from here.

**Grain:** one row per `(player_id, match_id)` for every player who actually
played (minutes > 0).

**How it is built:** for each competition we read the enriched events, group
them by `(match_id, player_id)`, and join the result onto the appearance
spine — which already carries `minutes_played` and the authoritative
`goals` / `assists` / `own_goals` / cards from the match sheet.

> **Goals are authoritative, not inferred.** `goals` comes from the lineup.
> `event_goals` (tag 101 on Shot/Free-Kick, excluding shootouts) is kept
> beside it only so we can confirm the two agree.

In [1]:
import sys, time
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import wyscout_lib as wl

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)
DATA = wl.DATA

appearances = pd.read_parquet(DATA / "player_appearances.parquet")
players = pd.read_parquet(DATA / "players_master.parquet")
print(f"appearances loaded: {appearances.shape}")

appearances loaded: (74098, 11)


## Build per competition, then concatenate
We process one competition at a time to keep memory flat — each enriched
event file is read, aggregated, joined, and released before the next.

In [2]:
t = time.time()
parts = []
for comp in wl.COMPETITIONS:
    ev = pd.read_parquet(DATA / "events_enriched" / f"{comp}.parquet")
    apps_c = appearances[appearances.competition_id == comp]
    pms_c = wl.build_player_match_stats(apps_c, ev)
    parts.append(pms_c)
    print(f"  {comp:14s} rows={len(pms_c):6,}  ({time.time()-t:5.1f}s)")
    del ev

pms = pd.concat(parts, ignore_index=True)
pms.to_parquet(DATA / "player_match_stats.parquet")
print(f"\nplayer_match_stats: {pms.shape}  written  ({time.time()-t:.1f}s)")
pms.head(3)

  england        rows=10,344  (  0.1s)
  france         rows=10,444  (  0.1s)
  germany        rows= 8,419  (  0.2s)


  italy          rows=10,518  (  0.3s)


  spain          rows=10,492  (  0.3s)
  euro_2016      rows= 1,394  (  0.3s)
  world_cup      rows= 1,769  (  0.3s)

player_match_stats: (53380, 42)  written  (0.4s)


,player_id,match_id,competition_id,team_id,started,minutes_played,subbed_on_minute,subbed_off_minute,goals,yellow_card,red_card,total_events,pass_total,pass_accurate,progressive_passes,final_third_passes,into_box_passes,key_passes,assists,smart_passes,crosses,crosses_accurate,shots_total,shots_on_target,event_goals,header_shots,header_goals,duels_total,defensive_duels,aerial_duels,attacking_duels,clearances,clearances_accurate,fouls,possession_retained,accelerations,touches,saves,reflexes,goal_kicks,goal_kicks_accurate,shot_distance_avg
0,9206,2500089,england,1646,True,61.0,NaN,61.0,1,0,0,35,13,7,2,9,2,1,0,0,0,0,1,1,1,0,0,17,0,8,4,0,0,2,2,0,1,0,0,0,0,8.062258
1,93,2500089,england,1646,True,80.0,NaN,80.0,0,0,0,51,27,21,6,17,8,0,0,1,8,3,0,0,0,0,0,7,0,2,3,0,0,1,4,0,6,0,0,0,0,NaN
2,10108,2500089,england,1646,True,90.0,NaN,NaN,0,0,0,72,45,39,14,8,1,0,0,0,0,0,1,0,0,1,0,19,9,8,1,2,0,1,5,0,3,0,0,0,0,10.440307


## Sanity: structure & coverage

In [3]:
print("rows (player-matches):", f"{len(pms):,}")
print("unique players       :", f"{pms.player_id.nunique():,}")
print("unique matches       :", f"{pms.match_id.nunique():,}")
print("columns              :", len(pms.columns))
print("\nminutes per appearance — describe:")
print(pms.minutes_played.describe()[["min","25%","50%","75%","max"]].round(1).to_string())
print("\nany player-match with events but 0 recorded minutes? ->",
      ((pms.minutes_played == 0)).sum(), "(must be 0)")

rows (player-matches): 53,380
unique players       : 3,020
unique matches       : 1,941
columns              : 42

minutes per appearance — describe:
min      1.0
25%     62.0
50%     90.0
75%     90.0
max    120.0

any player-match with events but 0 recorded minutes? -> 0 (must be 0)


## Validation — goals authoritative vs event-derived
These two columns are computed from completely different sources, so close
agreement is strong evidence the goal logic is correct.

In [4]:
g_auth = pms.goals.sum()
g_evt = pms.event_goals.sum()
print(f"Total goals (authoritative lineup): {g_auth:,}")
print(f"Total goals (event tag 101)       : {g_evt:,}")
print(f"agreement: {100*g_evt/g_auth:.1f}%")
print(f"Total assists (event tag 301): {pms.assists.sum():,}")
print("(own goals are not counted: FINISHING uses the lineup `goals` field, which excludes them)")

Total goals (authoritative lineup): 5,049
Total goals (event tag 101)       : 5,048
agreement: 100.0%
Total assists (event tag 301): 3,097
(own goals are not counted: FINISHING uses the lineup `goals` field, which excludes them)


## Validation — named players (ground truth)

In [5]:
def line(name, pid, comp=None):
    sel = pms[pms.player_id == pid]
    if comp:
        sel = sel.merge(pms_match_comp, on="match_id")
        sel = sel[sel.competition_id_m == comp]
    return {
        "player": name,
        "apps": len(sel),
        "minutes": int(sel.minutes_played.sum()),
        "goals": int(sel.goals.sum()),
        "assists": int(sel.assists.sum()),
        "shots": int(sel.shots_total.sum()),
        "saves": int(sel.saves.sum()),
    }

# helper to filter by competition
pms_match_comp = pms[["match_id"]].drop_duplicates().merge(
    pd.read_parquet(DATA / "matches_master.parquet")[["match_id", "competition_id"]],
    on="match_id").rename(columns={"competition_id": "competition_id_m"})

def pid_of(substr):
    hit = players[players.short_name.str.contains(substr, na=False, regex=False)]
    return hit.iloc[0].player_id if len(hit) else None

checks = pd.DataFrame([
    line("Harry Kane (all)", pid_of("H. Kane")),
    line("Harry Kane (WC)", pid_of("H. Kane"), "world_cup"),
    line("Mohamed Salah (all)", pid_of("Mohamed Salah") or pid_of("M. Salah")),
    line("Lionel Messi (all)", pid_of("L. Messi")),
    line("Cristiano Ronaldo", pid_of("Cristiano Ronaldo")),
    line("T. Courtois (WC)", pid_of("T. Courtois"), "world_cup"),
])
checks

,player,apps,minutes,goals,assists,shots,saves
0,Harry Kane (all),47,3896,35,2,183,0
1,Harry Kane (WC),6,573,6,0,11,0
2,Mohamed Salah (all),38,3102,34,8,141,0
3,Lionel Messi (all),40,3356,35,14,154,0
4,Cristiano Ronaldo,38,3282,33,7,196,0
5,T. Courtois (WC),7,630,0,0,0,35


**Expected:** Kane = 6 goals at the World Cup; Salah ~32 league goals; Messi &
Ronaldo elite league scorers; Courtois 35 saves at the World Cup. If these
line up, the atomic table is trustworthy and we can aggregate to careers.

Phase 2 complete → **Phase 3: `player_career_stats`**.